In [ ]:
!pip install importnb

In [ ]:
import sys
from importnb import Notebook

# Tell Python where the folder is (change this to your actual folder path)
sys.path.append("/content/drive/MyDrive/Colab Notebooks/Lora/")

# Now import it like a normal library
with Notebook():
    import Utils

In [ ]:
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from unsloth import FastLanguageModel
import torch

# 1. Load the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit", # Your exact test model
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters (This is the "surgical upgrade" part)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # The Rank. 16 is great for 150 documents.
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
from datasets import load_dataset

# 1. Load your uploaded file
dataset = load_dataset("json", data_files={"train": r"/content/drive/MyDrive/Colab Notebooks/Lora/training_set.jsonl"}, split="train")

# 2. Map the data into the Llama 3.2 Chat Template
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # This matches the Llama-3.2 Instruct format exactly
        text = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{instruction}<|eot_id|>" \
               f"<|start_header_id|>user<|end_header_id|>\n\n{input}<|eot_id|>" \
               f"<|start_header_id|>assistant<|end_header_id|>\n\n{output}<|eot_id|>"
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 80, # Seeing 150 docs once or twice
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# START THE TRAINING
trainer_stats = trainer.train()


In [ ]:
FastLanguageModel.for_inference(model)

def extract_pico(trial_text):
    instruction = "Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object."

    # Format the prompt exactly like the training data
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": trial_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Generate the extraction
    outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,        # Lower temperature = more focused/less creative
    repetition_penalty = 1.2 )

    # Decode and clean up the result
    # response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # json_part = response.split("assistant")[-1].strip()
    # return json_part
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}

# --- TEST IT WITH A NEW TRIAL ---
test_text = " Pre-operative short-term pulmonary rehabilitation for patients of chronic obstructive pulmonary disease undergoing coronary artery bypass graft surgery. The role of pre-operative short-term pulmonary rehabilitation in patients with chronic obstructive pulmonary disease who undergo coronary artery bypass graft surgery has been assessed for the first time prospectively. Forty-five patients posted for coronary artery bypass graft surgery were randomised to receive either short-term pulmonary rehabilitation (group I) or no such programme (group II). Patients of both the groups were evenly matched with respect to age, sex, body surface area, duration and severity of chronic obstructive pulmonary disease and coronary artery disease. Normal individuals who evenly matched with the study group were assessed for normal respiratory function parameters. Pre-operative and post-operative peak expiratory flow rate, inspiratory capacity, post-operative ventilation time, post-operative pulmonary complication and hospital stay were determined in both the groups. Peak expiratory flow rate (220.0 +/- 12.9 and 324.3 +/- 84.3 in group I, 218.0 +/- 16.4 and 260.5 +/- 35.2 in group II) and inspiratory capacity (844.0 +/- 147.4 and 1100.0 +/- 158.1 in group I, 830.0 +/- 117.4 and 1090 +/- 137 in group II) were significantly lower before and after surgery respectively in both groups compared to normal values. Even though both groups showed a significant rise in post-operative peak expiratory flow rate and inspiratory capacity after surgery, the post-operative peak expiratory flow rate and inspiratory capacity in group I was significantly higher than in group II. In group I, the post-operative ventilation time (24.5 +/- 6.00 hours), post-operative complications (n = 4) and hospital stay (12.4 +/- 3.6 days) were significantly lower than in group II (35.2 +/- 22.3 hours, n = 11, 18.8 +/- 6.6 days respectively). These data suggest that short-term pulmonary rehabilitation is feasible and effective in improving pulmonary functions before and after surgery and in reducing surgical morbidity and cost of medical care significantly."
print(extract_pico(test_text))

In [ ]:

model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

import shutil
shutil.copytree("lora_model", r"/content/drive/MyDrive/Colab Notebooks/Lora/medical_lora_model")


In [ ]:
from unsloth import FastLanguageModel

# Direct loading from your Drive folder
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/Colab Notebooks/Lora/medical_lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# Switch to inference mode
FastLanguageModel.for_inference(model)

In [ ]:
import json

In [ ]:
FastLanguageModel.for_inference(model)

def extract(trial_text):
    instruction = "Extract the Participants, Intervention, and Outcome from the trial text. Return the result strictly as a JSON object."

    # Format the prompt exactly like the training data
    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": trial_text},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    # Generate the extraction
    outputs = model.generate(
    input_ids = inputs,
    max_new_tokens = 512,
    use_cache = True,
    temperature = 0.1,        # Lower temperature = more focused/less creative
    repetition_penalty = 1.2 )

    # Decode and clean up the result
    # response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    # json_part = response.split("assistant")[-1].strip()
    # return json_part
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}



In [ ]:
file_path = r"/content/drive/MyDrive/Colab Notebooks/Lora/result_lora.csv"




In [ ]:
import pandas as pd
df = pd.read_excel(r"/content/drive/MyDrive/Colab Notebooks/cwk_data/cwk_train_data.xlsx")

In [ ]:
Utils.config(df, file_path, 200, extract)